In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

file_name = "IMDBDataset.csv"

# Search upward from current working directory and find the file by name
csv_path = None
for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    matches = list(root.rglob(file_name))
    if matches:
        csv_path = matches[0]
        break

if csv_path is None:
    raise FileNotFoundError(f"{file_name} not found in workspace search.")

df = pd.read_csv(csv_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
# text cleaning
import re
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove non-alphanumeric characters (except spaces)
    text = re.sub(r'[^A-Za-z0-9 ]+', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

df['review'] = df['review'].apply(clean_text)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [3]:
# remove stop words
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
def remove_stop_words(text):
    return ' '.join([word for word in text.split() if word not in ENGLISH_STOP_WORDS])  
df['review'] = df['review'].apply(remove_stop_words)
df.head()

,review,sentiment
0,reviewers mentioned watching just 1 oz episode...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [4]:
# apply stemming
%pip install nltk
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
def stem_text(text):
    return ' '.join([stemmer.stem(word) for word in text.split()])
df['review'] = df['review'].apply(stem_text)
df.head()


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


,review,sentiment
0,review mention watch just 1 oz episod youll ho...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [5]:
# vectorize the data
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['review'])
X.shape


(50000, 181778)

In [ ]:
y = df.iloc[:, -1].values
y.shape

(50000,)

: 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, CategoricalNB, BernoulliNB, GaussianNB
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Keep sparse input for models that support sparse matrices
model = MultinomialNB()
model3 = BernoulliNB()
model.fit(X_train, y_train)
model3.fit(X_train, y_train)

# Convert to dense for models that require dense arrays
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

model2 = CategoricalNB()
model4 = GaussianNB()
model2.fit(X_train_dense, y_train)
model4.fit(X_train_dense, y_train)

y_pred = model.predict(X_test)

print("Accuracy MultinomialNB:", accuracy_score(y_test, y_pred))
print("Accuracy CategoricalNB:", accuracy_score(y_test, model2.predict(X_test_dense)))
print("Accuracy BernoulliNB:", accuracy_score(y_test, model3.predict(X_test)))
print("Accuracy GaussianNB:", accuracy_score(y_test, model4.predict(X_test_dense)))
print("Classification Report:\n", classification_report(y_test, y_pred))